# S2S lead-time skill maps (weeks 1-2, 3-4, 5-6, 7-8)

## Purpose

Gridpoint ACC and normalized RMSE of two-week means of the five initialization configurations: the S2S analogue of the ESP-Lab lead-time skill maps (`ESP-Lab/jupyter/1a_*_leadtime_acc_skill_map.ipynb`, `1b_*_leadtime_rmse_skill_map.ipynb`), with lead windows of two weeks in place of lead seasons, and the December and May starts compared in one figure.

## Inputs

The weekly anomalies saved by `02_compute_and_plot_s2s_weekly_skill.ipynb` (`data/hind_lead_eval/s2s_weekly/anom_<BIAS>_<CONFIG>_<SEASON>_<VAR>.nc`: ensemble-mean and observed anomalies per start and week, for each bias treatment); run notebook 02 first. A file is used only when it was built from the current case of its configuration and with the `CLIM_YEARS` and bias treatment set here.

## Method

`util/fcst_init_skill.py` (`window_skill`, `window_region_summary`):

1. The weekly anomalies are averaged over each window of `WINDOWS` (weeks 1-2 = lead days 1-14, ..., weeks 7-8 = days 43-56).
2. Per gridpoint, over the 4 starts of a season (and, with `WINDOW_POOLED`, the 8 starts of both seasons, each anomaly from its own season's climatology):
   * ACC: Pearson correlation of the ensemble-mean and observed anomalies, stippled above the one-sided 95% critical value (0.90 for 4 starts, 0.62 for 8); the panel number is the area above it (about 5% by chance).
   * nRMSE = RMSE of the ensemble-mean anomaly / RMSE of the climatology forecast (the RMS of the observed anomaly); below 1 the forecast beats climatology (nRMSE² = 1 − MSSS at the gridpoint); the panel number is the area below 1. ESP-Lab divides by the standard deviation of its detrended observed anomalies; with anomalies from a multi-year climatology and only 4 starts the RMS is the matching scale. Cells whose climatology-forecast RMSE is below `MIN_OBS_FRACTION` of its map median are left out (ESP-Lab `min_obs_std_fraction`), e.g. dry desert precipitation.
3. Differences from `BASELINE` (ΔACC, ΔnRMSE) on the same starts; the panel numbers are the areas where the configuration is better / worse.
4. Regional summary per region and window: cos-lat weighted mean ACC, significant area, regional nRMSE sqrt(Σw RMSE² / Σw RMSE_clim²) and the area beating climatology.

With 4 samples a single gridpoint ACC is very noisy: read the maps for coherent large-scale patterns and the area numbers, not single points.

Caveat: ERA5 is the verifying analysis and the source of the HYB_BASE/DA_LND atmospheric initial state, which favours them in weeks 1-2; their current initial files also lack the radiation gases (cold start over land), so their columns are invalid until those runs are redone.

## Outputs

`data/hind_lead_eval/s2s_leadtime_maps/s2s_window_summary.csv` (regional summary); figures in `figure/hind_lead_eval/s2s_leadtime_maps/`: `s2s_window_<METRIC>_<BIAS>_<VAR>.png`, METRIC = `acc`, `nrmse`, `dacc`, `dnrmse` (seasons stacked, windows in rows, configurations in columns). Only the selection of section 7 is plotted.

## 1. Libraries

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display

## 2. User setup

Paths and options. Edit this cell for normal changes; it holds plain values only, so `papermill -p` overrides work. `CONFIGS`, `CLIM_YEARS` and `BIASES` must match the run of notebook 02 that saved the anomalies.

In [ ]:
WORK_DIR = Path("/global/cfs/cdirs/m4849/zhan391/work_package/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart/diag_dart_2026")
OBSERVATION_ROOT = Path("/global/cfs/cdirs/e3sm/zhan391/data")  # reference-data archive
LAND_SUPPORT_DIR = OBSERVATION_ROOT / "lnd_sea_mask"           # land fraction (land/ocean regions)
WORKFLOW_NAME = "hind_lead_eval"

# Initialization configurations (configs/fcst_init_experiment_config.py): configuration key: raw case name
CONFIGS = {
    "CLIM_BASE": "F20TR_ne30pg2_r05_IcoswISC30E3r5_CLIM_BASE",
    "HYB_BASE": "F20TR_ne30pg2_r05_IcoswISC30E3r5_HYB_BASE",
    "DA_ATM": "F20TR_ne30pg2_r05_IcoswISC30E3r5_DA_ATM",
    "DA_LND": "F20TR_ne30pg2_r05_IcoswISC30E3r5_DA_LND",
    "DA_COUPLED": "F20TR_ne30pg2_r05_IcoswISC30E3r5_DA_COUPLED",
}
SEASONS = ("DJF", "JJA")
VARIABLES = ("Z500", "T850", "U850", "U200", "TMQ", "PSL", "TREFHT", "PRECT", "FLUT")
BIASES = ("none", "amip")  # bias treatments of notebook 02 ("none": vs observed climatology; "amip": AMIP climatology)
CLIM_YEARS = (2001, 2014)  # climatology years of notebook 02's anomalies

WINDOWS = ((1, 2), (3, 4), (5, 6), (7, 8))   # weeks of each lead window
BASELINE = "CLIM_BASE"     # reference of the difference maps: the free-running state (no initialization information)
MIN_OBS_FRACTION = 0.01    # nRMSE left out where the climatology RMSE < this x its map median

# Regions of the summary table: standard names (configs/s2s_config.py): GLOBAL, OCEAN, LAND, NINO3.4, MAR_CONT,
# CONUS, NH_POLAR; add your own boxes to CUSTOM_REGIONS (lat -90..90, lon 0-360, lon0 > lon1 wraps through 0E,
# mask None, "land" or "ocean") and name them in REGIONS.
CUSTOM_REGIONS = {
    # "EUROPE": dict(lat=(35, 72), lon=(350, 40), mask="land"),
}
REGIONS = ("GLOBAL", "LAND", "OCEAN", "NH_POLAR", "NINO3.4", "MAR_CONT", "CONUS")

## 3. Paths and modules

In [ ]:
repo = WORK_DIR.resolve()
if str(repo / "diagnostic") not in sys.path:
    sys.path.insert(0, str(repo / "diagnostic"))
from configs.fcst_init_experiment_config import cache_current, check_configs, config_label, set_case_names
from configs.output_paths import workflow_output_dirs
from util.fcst_init_data import region_label, region_masks, regions_table, select_regions

check_configs(CONFIGS)
set_case_names(CONFIGS)   # cache_current compares each saved anomaly file with these case names
if BASELINE not in CONFIGS:
    raise ValueError(f"BASELINE {BASELINE!r} is not in CONFIGS")
anom_dir, _ = workflow_output_dirs(WORKFLOW_NAME, "s2s_weekly", output_root=DIAGNOSTIC_OUTPUT_ROOT, create=False)
data_dir, figure_dir = workflow_output_dirs(WORKFLOW_NAME, "s2s_leadtime_maps", output_root=DIAGNOSTIC_OUTPUT_ROOT)
print(f"[INPUT] {anom_dir}\n[DATA] {data_dir}\n[FIGURE] {figure_dir}")

regions = select_regions(REGIONS, CUSTOM_REGIONS)
REGION_LABELS = {r["name"]: region_label(r) for r in regions}
display(regions_table(regions))


def save(fig, name):
    path = figure_dir / name
    fig.savefig(path, dpi=FIGURE_DPI, bbox_inches="tight")
    print(f"[FIGURE] {path}")
    if not SHOW:
        plt.close(fig)

## 4. Read input

The weekly anomalies of notebook 02. A missing file, or one built from another case, other climatology years or another bias treatment, is listed and skipped: rerun notebook 02 for it.

In [ ]:
anom, skipped = {}, []
for b in BIASES:
    for c in CONFIGS:
        for s in SEASONS:
            for v in VARIABLES:
                path = anom_dir / f"anom_{b}_{c}_{s}_{v}.nc"
                if not cache_current(path, c):
                    skipped.append((b, c, s, v, "missing" if not path.exists() else "other case"))
                    continue
                ds = xr.load_dataset(path)
                if ds.attrs.get("bias") != b or ds.attrs.get("clim_years") != str(CLIM_YEARS):
                    skipped.append((b, c, s, v, f"bias {ds.attrs.get('bias')}, years {ds.attrs.get('clim_years')}"))
                    continue
                anom[b, c, s, v] = ds
print(f"[LOAD] {len(anom)} anomaly files from {anom_dir}")
if skipped:
    print(f"[SKIP] {len(skipped)} not available (run notebook 02 for them):")
    display(pd.DataFrame(skipped, columns=["bias", "config", "season", "variable", "reason"]))
if not anom:
    raise FileNotFoundError(f"no current anomaly files in {anom_dir}: run 02_compute_and_plot_s2s_weekly_skill.ipynb")

## 5. Compute and save

Window skill maps per bias treatment, configuration, season and variable, and of both seasons pooled (`"ALL"`, when a configuration has both); their regional summary is saved. The maps are kept in memory (a few seconds to recompute).

In [ ]:
from util.fcst_init_skill import pool_seasons, window_region_summary, window_skill

win = {}
for b in BIASES:
    for c in CONFIGS:
        for v in VARIABLES:
            have = [s for s in SEASONS if (b, c, s, v) in anom]
            for s in have:
                win[b, c, s, v] = window_skill(anom[b, c, s, v].model, anom[b, c, s, v].reference, WINDOWS,
                                               min_obs_fraction=MIN_OBS_FRACTION)
            if len(have) > 1:
                win[b, c, "ALL", v] = window_skill(pool_seasons({s: anom[b, c, s, v].model for s in have}),
                                                   pool_seasons({s: anom[b, c, s, v].reference for s in have}),
                                                   WINDOWS, min_obs_fraction=MIN_OBS_FRACTION)
first = next(iter(anom.values()))
masks = region_masks(regions, first.lat.values, first.lon.values, land_support_dir=LAND_SUPPORT_DIR)
summary = pd.concat([window_region_summary(sk, masks).assign(bias=b, config=c, season=s, variable=v)
                     for (b, c, s, v), sk in win.items()], ignore_index=True)
summary.to_csv(data_dir / "s2s_window_summary.csv", index=False)
print(f"[WRITE] {data_dir / 's2s_window_summary.csv'}  ({len(win)} window-skill maps)")
show = summary[(summary.region == REGIONS[0]) & (summary.bias == BIASES[-1])]
display(show.pivot_table(index=["variable", "config"], columns=["season", "window"], values=["acc_mean", "nrmse_region"],
                         sort=False).round(2))

## 6. Plot and show

Only the selection at the top of the next cell is plotted. Each figure stacks the seasons (Dec and May starts; with `WINDOW_POOLED` also both pooled), windows in rows and configurations in columns, on one colour scale. Change the settings and rerun that cell alone.

In [ ]:
# ============================ Figure settings: edit here and rerun this cell ============================
PLOT_BIASES = ("amip",)                       # subset of BIASES
PLOT_SEASONS = ("DJF", "JJA")                 # sections, top to bottom
PLOT_VARIABLES = ("TREFHT", "PRECT", "Z500")  # subset of VARIABLES
PLOT_METRICS = ("acc", "nrmse", "dacc", "dnrmse")   # acc, nrmse; dacc, dnrmse = minus BASELINE
WINDOW_POOLED = False      # True: add a section of both seasons pooled (8 starts, ACC critical value 0.62)
MAP_EXTENT = None          # (lon0, lon1, lat0, lat1) in degrees to zoom, e.g. (235, 294, 24, 50) for CONUS; None: globe
MAP_PANEL_SIZE = (None, None)   # (width, height) of one panel in inches; None: from FONT_SIZE
METRIC_STYLE = {   # metric: colour levels, colour map, colour-bar label, colour-bar extension
    "acc": (np.linspace(-1, 1, 21), "RdBu_r", "ACC", "neither"),
    "nrmse": (np.linspace(0, 2, 21), "RdBu_r", "nRMSE (RMSE / climatology RMSE)", "max"),
    "dacc": (np.linspace(-1, 1, 21), "RdBu_r", "ΔACC", "both"),
    "dnrmse": (np.linspace(-1, 1, 21), "RdBu_r", "ΔnRMSE", "both"),
}
SHOW = True          # False: save the figures without showing them
FIGURE_DPI = 200
FONT_SIZE = 18       # base font size (pt); every label of every figure scales with it
FONT_SCALES = dict(label=1.0, title=1.0, tick=0.9, legend=1.0, suptitle=1.15, cell=0.75)  # x FONT_SIZE

unknown = [(k, x) for k, xs, ok in (("bias", PLOT_BIASES, BIASES), ("season", PLOT_SEASONS, SEASONS),
                                    ("variable", PLOT_VARIABLES, VARIABLES), ("metric", PLOT_METRICS, METRIC_STYLE))
           for x in xs if x not in ok]
if unknown:
    raise ValueError(f"not available: {unknown}")

from util.fcst_init_plotting import set_font_size
set_font_size(FONT_SIZE, **FONT_SCALES)
# =========================================================================================================

from util.fcst_init_plotting import plot_window_maps
from util.fcst_init_skill import area_fraction, critical_acc


def sections(b, v, configs):
    """{season: {config: window skill}} of the plotted seasons (and both pooled)."""
    names = list(PLOT_SEASONS) + (["ALL"] if WINDOW_POOLED and len(PLOT_SEASONS) > 1 else [])
    out = {s: {c: win[b, c, s, v] for c in configs if (b, c, s, v) in win} for s in names}
    return {s: x for s, x in out.items() if x}


def draw(metric, b, v, panels, title, stipple=None, note=None):
    levels, cmap, label, extend = METRIC_STYLE[metric]
    save(plot_window_maps(panels, title=title, levels=levels, cmap=cmap, label=label, extend=extend, extent=MAP_EXTENT,
                          panel_size=MAP_PANEL_SIZE, stipple=stipple, note=note), f"s2s_window_{metric}_{b}_{v}.png")


for b in PLOT_BIASES:
    for v in PLOT_VARIABLES:
        secs = sections(b, v, CONFIGS)
        if not secs:
            print(f"[SKIP] {v}, BIAS = {b}: no anomalies")
            continue
        if "acc" in PLOT_METRICS:
            acc = {s: {c: sk.acc for c, sk in x.items()} for s, x in secs.items()}
            sig = {s: {c: sk.acc > sk.attrs["acc_critical_95"] for c, sk in x.items()} for s, x in secs.items()}
            n = sorted({sk.attrs["n_samples"] for x in secs.values() for sk in x.values()})
            crit = ", ".join(f"{critical_acc(k):.2f} ({k} starts)" for k in n)
            draw("acc", b, v, acc, f"{v} gridpoint ACC of two-week means, BIAS = {b} (stippled > {crit})", sig,
                 lambda s, c, w: f"{area_fraction(sig[s][c].sel(window=w), acc[s][c].sel(window=w).notnull()):.0f}%")
        if "nrmse" in PLOT_METRICS:
            nr = {s: {c: sk.nrmse for c, sk in x.items()} for s, x in secs.items()}
            draw("nrmse", b, v, nr, f"{v} normalized RMSE of two-week means, BIAS = {b} (< 1 beats climatology)",
                 note=lambda s, c, w: f"<1: {area_fraction(nr[s][c].sel(window=w) < 1, nr[s][c].sel(window=w).notnull()):.0f}%")
        others = {s: {c: sk for c, sk in x.items() if c != BASELINE} for s, x in secs.items() if BASELINE in x}
        for m, sign in (("acc", 1), ("nrmse", -1)):   # sign: + where higher is better
            if "d" + m not in PLOT_METRICS or not any(others.values()):
                continue
            d = {s: {c: sk[m] - secs[s][BASELINE][m] for c, sk in x.items()} for s, x in others.items()}
            what = "ACC" if m == "acc" else "nRMSE"
            draw("d" + m, b, v, d, f"{v} {what} minus {config_label(BASELINE)}, two-week means, BIAS = {b} "
                                   f"(numbers: area better / worse)",
                 note=lambda s, c, w, d=d, sign=sign: (f"{area_fraction(sign * d[s][c].sel(window=w) > 0):.0f}% / "
                                                       f"{area_fraction(sign * d[s][c].sel(window=w) < 0):.0f}%"))